In [1]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

In [2]:
model_path = "/mnt/storage_C1/igorzwirtes/poster_ic/qwen2.5coder"
adapter_path = "./qwen-spider-finetuned/r64b4a128"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Avaliação sem LoRA
'''
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(model_path, quantization_config=bnb_config, device_map="auto")
model.eval()
'''

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [5]:
dataset = load_dataset("spider")
test_data = dataset["validation"]

In [ ]:
# Avaliação sem LoRA
'''
def build_prompt(example):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert SQL assistant. "
                "Output ONLY the SQL query with no explanation, "
                "no markdown, no backticks, no comments. "
                "Just the raw SQL query ending with a semicolon."
            )
        },
        {
            "role": "user",
            "content": f"Database: {example['db_id']}\nQuestion: {example['question']}"
        }
    ]
    
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
'''

In [6]:
def build_prompt(example):
    messages = [
        {
            "role": "system",
            "content": "You are an expert SQL assistant. Given a natural language question and a database schema, generate the correct SQL query."
        },
        {
            "role": "user",
            "content": f"Database: {example['db_id']}\nQuestion: {example['question']}"
        }
    ]
    
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
predictions = []

for example in test_data:
    prompt = build_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=900).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,    
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decodifica só os tokens novos
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = tokenizer.decode(generated, skip_special_tokens=True).strip()

    predictions.append({
        "db_id": example["db_id"],
        "question": example["question"],
        "gold": example["query"],
        "predicted": pred_sql,
    })
    print(f"Q: {example['question']}\nGold: {example['query']}\nPred: {pred_sql}\n---")

with open("predictions.json", "w") as f:
    json.dump(predictions, f, indent=2)

print(f"\nPredições salvas: {len(predictions)} exemplos")

Q: How many singers do we have?
Gold: SELECT count(*) FROM singer
Pred: SELECT COUNT(*) FROM singer;
---
Q: What is the total number of singers?
Gold: SELECT count(*) FROM singer
Pred: SELECT COUNT(*) FROM singer;
---
Q: Show name, country, age for all singers ordered by age from the oldest to the youngest.
Gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Pred: SELECT name, country, age FROM concert_singer ORDER BY age DESC;
---
Q: What are the names, countries, and ages for every singer in descending order of age?
Gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Pred: SELECT name, country, TIMESTAMPDIFF(YEAR, birth_date, CURDATE()) AS age FROM concert_singer ORDER BY age DESC;
---
Q: What is the average, minimum, and maximum age of all singers from France?
Gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
Pred: SELECT MIN(age) AS min_age, MAX(age) AS max_age, AVG(age) AS avg_age FROM singer WHERE country = 'France';
